# Composing Indexes from Separately Built Parts

A two-stage index is a first stage that retrieves candidates and a rescoring dataset that reorders them. The two are independent: any first stage can be combined with any rescoring dataset that describes the same collection.

`kannolo.compose(index, rerank=...)` combines them in memory. The first stage is built (or loaded) on its own. The rescoring stage is built from an array, the same input that `kannolo.build` takes. The composed index **shares** the first stage and does not copy it. One loaded first stage can be composed with several rescoring datasets.

You store each part once: three first stages and one collection are four files. Rescored with three encoders, they give nine indexes. This notebook applies this to a dense collection (SIFT1M), then shows the same pattern for a hybrid pipeline: a sparse first stage with multi-vector rescoring.

## Imports

In [ ]:
import importlib.metadata as md

print("kannolo", md.version("kannolo"))

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import kannolo

## Load the Collection

In [ ]:
# A dense collection as 2-D float32 arrays, e.g. SIFT1M (128 dimensions, squared Euclidean)
data_path = ""          # (n, dim) base vectors
queries_path = ""       # (n_queries, dim) queries
groundtruth_path = ""   # optional: (n_queries, >= k) exact neighbor ids; computed below if empty
metric = "l2"

work_dir = Path("compose_data")  # where the first stages are saved
work_dir.mkdir(exist_ok=True)

k = 10
n_queries = 1000  # queries used for recall and timing

data = np.load(data_path).astype(np.float32)
queries = np.load(queries_path).astype(np.float32)[:n_queries]
print(f"Collection {data.shape}, queries {queries.shape}")

if groundtruth_path:
    groundtruth = np.load(groundtruth_path)[:n_queries, :k]
else:
    start = time.time()
    _, groundtruth = kannolo.build(data, metric=metric, index=kannolo.Flat()).search(queries, k)
    print(f"Exact ground truth computed in {time.time() - start:.1f} s")


def recall(ids, truth=None):
    truth = groundtruth if truth is None else truth
    return np.mean([len(set(r[r >= 0]) & set(g)) / len(g) for r, g in zip(ids, truth)])

## Build and Save Three First Stages

Each first stage is a complete index on its own, saved to its own file. They store the collection in compressed form (1-bit RaBitQ, PQ, 4-bit RaBitQ residuals), so they are fast but approximate. The rescoring stage restores the accuracy.

In [ ]:
first_stages = {
    "HNSW RaBitQ(1)": dict(encoder=kannolo.RaBitQ(bits=1), index=kannolo.HNSW(m=32, ef_construction=200, graph="permuted")),
    "HNSW PQ(32)": dict(encoder=kannolo.PQ(subspaces=32), index=kannolo.HNSW(m=32, ef_construction=200, graph="compressed")),
    "IVF RaBitQ(4)": dict(encoder=kannolo.RaBitQ(bits=4), index=kannolo.IVF(n_lists=4096, residual=True)),
}
# Each first stage's search parameters (1-bit RaBitQ works best with 4-bit queries)
first_params = {
    "HNSW RaBitQ(1)": dict(ef_search=100, query_bits=4),
    "HNSW PQ(32)": dict(ef_search=100),
    "IVF RaBitQ(4)": dict(nprobe=32),
}

for name, kwargs in first_stages.items():
    path = work_dir / (name.replace(" ", "_").replace("(", "").replace(")", "") + ".index")
    if not path.exists():
        start = time.time()
        kannolo.build(data, metric=metric, **kwargs).save(path)
        print(f"{name:<15} built and saved in {time.time() - start:6.1f} s")
    first_stages[name] = path
    print(f"{name:<15} {path.stat().st_size / 2**20:8.1f} MiB on disk")

## Compose Them with Three Rescoring Stages

The rescoring data is the collection itself, kept as an array. Each rescoring stage encodes it at compose time, as exact `f32`, as `f16`, or as 8-bit RaBitQ codes.

Every first stage is loaded once and composed three times. With exact (`f32`) rescoring, a two-stage index finds every true neighbor that is among its first stage's candidates. The cheaper rescoring encodings come close.

In [ ]:
rescorings = {"f32": kannolo.Plain("f32"), "f16": kannolo.Plain("f16"), "RaBitQ(8)": kannolo.RaBitQ(bits=8)}
k_candidates = 100

rows = []
for name, path in first_stages.items():
    first = kannolo.load(path)  # loaded once, shared by the three composed indexes
    params = first_params[name]
    candidates = first.search(queries, k_candidates, **params)[1]
    coverage = recall(candidates)  # true top-k among the candidates
    first_recall = recall(first.search(queries, k, **params)[1])
    for rname, encoder in rescorings.items():
        start = time.time()
        index = kannolo.compose(first, kannolo.Rerank(encoder, data=data))
        compose_s = time.time() - start
        index.search(queries[:10], k, k_candidates=k_candidates, **params)  # warm-up
        start = time.time()
        _, ids = index.search(queries, k, k_candidates=k_candidates, num_threads=1, **params)
        qps = len(queries) / (time.time() - start)
        rows.append((name, rname, compose_s, index.nbytes / 2**20, first_recall, recall(ids), coverage, qps))

print(f"{'first stage':<15} {'rescoring':<10} {'compose':>8} {'memory':>9} {'first only':>10} {'two-stage':>9} {'candidates':>10} {'QPS':>7}")
for name, rname, compose_s, mib, r1, r, cov, qps in rows:
    print(f"{name:<15} {rname:<10} {compose_s:>7.1f}s {mib:>6.0f} MiB {r1:>10.3f} {r:>9.3f} {cov:>10.3f} {qps:>7.0f}")

- **first only**: recall@10 of the first stage alone.
- **two-stage**: recall after rescoring the first stage's `k_candidates` candidates.
- **candidates**: the fraction of the true top-10 among those candidates. Exact rescoring reaches this value.
- **memory**: the size of the composed index as a whole. The first stage is shared: this notebook holds each one in memory once, although it is part of three indexes.
- To keep a combination as a single file, save the composed index: it writes one file with both stages.

## The Same Pattern for a Hybrid Pipeline

The rescoring stage can use a different representation from the first stage, with its own data. Here a sparse first stage (learned sparse embeddings) is composed with multi-vector rescoring (late interaction, MaxSim). Fill in the paths to run this example; otherwise it is skipped.

In [ ]:
sparse_documents_path = ""  # sparse documents, seismic .bin format
sparse_queries_path = ""    # folder with components.npy, values.npy, offsets.npy
multivec_tokens_path = ""   # (n_tokens, dim) token vectors of all documents, concatenated
multivec_doclens_path = ""  # (n_documents,) tokens per document
multivec_queries_path = ""  # (n_queries, n_query_tokens, dim) query token vectors

if sparse_documents_path:
    sparse_first = kannolo.build(sparse_documents_path, encoder=kannolo.DotPacking8(), index=kannolo.HNSW(m=32))
    sparse_first.save(work_dir / "sparse_first.index")  # stored once, composed at will

    tokens, doclens = np.load(multivec_tokens_path).astype(np.float32), np.load(multivec_doclens_path)
    hybrid = kannolo.compose(sparse_first, kannolo.Rerank(kannolo.TwoLevelPQ(subspaces=16), data=(tokens, doclens)))
    print(hybrid)

    sparse_queries = tuple(np.load(Path(sparse_queries_path) / f"{a}.npy") for a in ("components", "values", "offsets"))
    multivec_queries = list(np.load(multivec_queries_path).astype(np.float32))
    dists, ids = hybrid.search(sparse_queries, k, rerank_queries=multivec_queries, k_candidates=100)
    print(ids[:3])
else:
    print("Hybrid example skipped: set the paths above to run it")